# Taller formativo - Inspección del dataset de vocales (ML)

**Asignatura:** Inteligencia Artificial - Universidad del Magdalena
**Curso:** IA-2026-II  |  **Tema:** Machine Learning - Reconocimiento óptico de caracteres (OCR)

## Contenido del equipo
- Equipos asignados: vocales **A, E, I, O, U**.
- La vocal **O** se representa con la carpeta **`0`** (no existe directorio propio para la letra O, por similitud visual con el dígito cero).

## Alcance de este archivo
Este notebook está normalizado para analizar **únicamente las vocales**. Se excluyen los números del 1 al 9 y cualquier otra letra que no sea vocal. Los conteos y estadísticas se reportan sobre el subconjunto de vocales.


## 1. Descripción del problema

El reconocimiento óptico de caracteres (OCR) permite identificar caracteres presentes en imágenes y convertirlos en texto digital editable. En este trabajo el conjunto de datos contiene imágenes manuscritas de caracteres, y el equipo se enfoca en las cinco vocales: **A, E, I, O, U**, donde la **O** se aproxima mediante el directorio del dígito `0`.

Esta decisión debe tenerse en cuenta al interpretar los resultados, ya que la letra O y el número 0 son caracteres distintos aunque presenten formas similares.


## 2. Configuración y carga del dataset

Se configura la ruta del dataset y se cargan solo las carpetas de clase correspondientes a vocales. El dataset completo (**CNN letter Dataset**) no se incluye en este repositorio por su peso.


In [ ]:
import os
import re
import zipfile
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)
np.random.seed(42)
plt.rcParams["figure.dpi"] = 100

# ---------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------
# Ruta del dataset (ajustar si es necesario). El dataset no se sube al repo por su peso.
RUTA_DATASET = r"CNN letter Dataset"
RUTA_ZIP = "archive (1).zip"
EXT = {".png", ".jpg", ".jpeg", ".bmp", ".gif", ".tif", ".tiff"}

# Vocales que analiza este grupo. Se excluyen números 1-9 y otras letras.
VOCALES = ["A", "E", "I", "O", "U"]

def _tiene_clases(carpeta: Path) -> bool:
    subs = [d for d in carpeta.iterdir() if d.is_dir() and not d.name.startswith(("__", "."))]
    con_imgs = [d for d in subs if any(f.suffix.lower() in EXT for f in d.iterdir() if f.is_file())]
    return len(con_imgs) >= 2

def encontrar_raiz() -> Path:
    p = Path(RUTA_DATASET)
    if p.is_dir() and _tiene_clases(p):
        return p
    candidatos_zip = [Path(RUTA_ZIP), Path("/mnt/user-data/uploads") / RUTA_ZIP]
    for z in candidatos_zip:
        if z.is_file():
            destino = Path("dataset_extraido")
            if not destino.exists():
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(destino)
            break
    for base in [Path("dataset_extraido"), Path(".")]:
        if base.is_dir():
            for d in [base] + [x for x in base.rglob("*") if x.is_dir()]:
                if "__MACOSX" in d.parts:
                    continue
                if _tiene_clases(d):
                    return d
    raise FileNotFoundError(
        "No se encontró el dataset. Ajuste RUTA_DATASET o RUTA_ZIP en esta celda."
    )

RAIZ = encontrar_raiz()
print("Dataset encontrado en:", RAIZ.resolve())


## 3. Organización de archivos y carpetas

Se muestra el árbol de carpetas restringido a las clases de vocales. Cada subcarpeta es una clase.


In [ ]:
for carpeta, subs, archivos in os.walk(RAIZ):
    if "__MACOSX" in carpeta:
        continue
    nombre = Path(carpeta).name
    # Solo mostrar carpetas de clase vocales o la raíz
    if carpeta == str(RAIZ) or nombre in VOCALES or nombre == "0":
        nivel = len(Path(carpeta).relative_to(RAIZ).parts)
        print("  " * nivel + f"{nombre}/  ({len(archivos)} archivos)")

print("\nEjemplos de nombres de archivo por vocal:")
for d in sorted(x for x in RAIZ.iterdir() if x.is_dir() and (x.name in VOCALES or x.name == "0")):
    print(f"  {d.name}: {[f.name for f in sorted(d.iterdir())[:3]]}")


## 4. Cantidad de imágenes por clase

Se construye una tabla con la ruta, clase y formato de cada imagen. La carpeta `0` se mapea a la vocal `O`. Se excluyen del análisis las carpetas de números del 1 al 9 y demás letras que no sean vocales.


In [ ]:
registros = []
for p in RAIZ.rglob("*"):
    if p.is_file() and p.suffix.lower() in EXT and "__MACOSX" not in p.parts:
        clase = p.parent.name
        # Mapear carpeta '0' a vocal 'O'
        if clase == "0":
            clase = "O"
        # Filtrar solo vocales (excluir números 1-9 y otras letras)
        if clase in VOCALES:
            registros.append({"ruta": p, "clase": clase, "formato": p.suffix.lower()})

df = pd.DataFrame(registros)
CLASES = sorted(df["clase"].unique())

print("Total de imágenes (vocales):", len(df))
print("Clases (vocales):", CLASES)

conteo = df["clase"].value_counts().reindex(VOCALES).fillna(0).astype(int)
print("\nConteo por vocal:")
print(conteo.to_string())
print("\nTotal:", int(conteo.sum()))
print("Razón entre la clase más y menos frecuente:", round(conteo.max() / conteo.min(), 3))

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(conteo.index, conteo.values, color="steelblue", edgecolor="black")
ax.set_title("Número de imágenes por vocal")
ax.set_xlabel("Clase (vocal)")
ax.set_ylabel("Cantidad de imágenes")
ax.set_ylim(0, conteo.max() * 1.1)
for b, v in zip(barras, conteo.values):
    ax.text(b.get_x() + b.get_width() / 2, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()
df.head()


## 5. Muestras aleatorias por clase

Se seleccionan imágenes aleatorias de cada vocal y se muestran en una cuadrícula para observar variaciones de escritura, fondo, iluminación, grosor de trazo, rotación o ruido.


In [ ]:
N_POR_CLASE = 8
rng = np.random.RandomState(7)

fig, axes = plt.subplots(len(CLASES), N_POR_CLASE, figsize=(2 * N_POR_CLASE, 2.3 * len(CLASES)))
for i, c in enumerate(CLASES):
    sub = df[df["clase"] == c]
    idx = rng.choice(len(sub), size=min(N_POR_CLASE, len(sub)), replace=False)
    for j, k in enumerate(idx):
        with Image.open(sub.iloc[k]["ruta"]) as im:
            a = np.array(im)
        ax = axes[i, j]
        ax.imshow(a, cmap="gray", vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(f"Clase {c}", fontsize=12, rotation=0, labelpad=35, va="center")
fig.suptitle("Imágenes aleatorias de cada vocal", fontsize=14)
plt.tight_layout()
plt.show()


## 6. Dimensiones, canales y formato

Se verifican todas las imágenes del subconjunto de vocales: tamaño, modo de color, número de canales, tipo de dato y rango de intensidades. Se determina si todas tienen el mismo tamaño.


In [ ]:
filas = []
for ruta, clase in zip(df["ruta"], df["clase"]):
    try:
        with Image.open(ruta) as im:
            a = np.array(im)
            filas.append({
                "clase": clase,
                "ancho": im.size[0],
                "alto": im.size[1],
                "modo": im.mode,
                "canales": 1 if a.ndim == 2 else a.shape[2],
                "dtype": str(a.dtype),
                "min": int(a.min()),
                "max": int(a.max()),
                "media": float(a.mean()),
                "desv": float(a.std()),
            })
    except Exception as e:
        filas.append({"clase": clase, "ancho": np.nan, "alto": np.nan, "modo": "ERROR",
                      "canales": np.nan, "dtype": "", "min": np.nan, "max": np.nan,
                      "media": np.nan, "desv": np.nan})

info = pd.DataFrame(filas)
info["ruta"] = df["ruta"].values

print("Tamaños (ancho, alto) y frecuencia:")
print(info.groupby(["ancho", "alto"]).size().to_string())
print("\nModos de color:\n", info["modo"].value_counts().to_string())
print("\nNúmero de canales:\n", info["canales"].value_counts().to_string())
print("\nTipo de dato:\n", info["dtype"].value_counts().to_string())
print("\nFormato de archivo:\n", df["formato"].value_counts().to_string())

mismo_tamano = info[["ancho", "alto"]].drop_duplicates().shape[0] == 1
print("\n¿Todas las imágenes tienen el mismo tamaño?", mismo_tamano)


## 7. Calidad de los datos

Se revisa: imágenes corruptas o ilegibles, duplicados exactos por contenido, imágenes idénticas en clases distintas, brillo/contraste atípicos y el patrón de los nombres de archivo.


In [ ]:
# 1) Imágenes corruptas o ilegibles
corruptas = []
for ruta in df["ruta"]:
    try:
        with Image.open(ruta) as im:
            im.verify()
    except Exception:
        corruptas.append(ruta)
print("Imágenes corruptas:", len(corruptas))

# 2) Duplicados exactos por contenido (hash MD5)
def md5(ruta):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

df["hash"] = [md5(r) for r in df["ruta"]]
print("Archivos duplicados exactos:", int(df["hash"].duplicated().sum()))
print("Contenidos repetidos en clases distintas:", int((df.groupby("hash")["clase"].nunique() > 1).sum()))

# 3) Valores faltantes en la tabla de metadatos
print("\nValores faltantes en la tabla de metadatos:")
print(df[["ruta", "clase", "formato"]].isna().sum().to_string())

# 4) Imágenes casi uniformes (poco contraste) o muy oscuras/claras
UMBRAL_DESV = 10
poco_contraste = info[info["desv"] < UMBRAL_DESV]
print(f"\nImágenes con desviación estándar < {UMBRAL_DESV} (poco contraste): {len(poco_contraste)}")
if len(poco_contraste):
    print(poco_contraste["clase"].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for c in CLASES:
    axes[0].hist(info.loc[info["clase"] == c, "media"], bins=30, alpha=0.5, label=c)
axes[0].set_title("Brillo medio por imagen, según clase")
axes[0].set_xlabel("Intensidad media del píxel (0-255)")
axes[0].set_ylabel("Número de imágenes")
axes[0].legend(title="Clase")

info.boxplot(column="desv", by="clase", ax=axes[1])
axes[1].set_title("Contraste (desv. estándar) por clase")
axes[1].set_xlabel("Clase")
axes[1].set_ylabel("Desviación estándar de píxeles")
plt.suptitle("")
plt.tight_layout()
plt.show()

# 5) Patrón de los nombres de archivo
patron = re.compile(r"^aug(\d+)_(\d+)$")
nombres = df["ruta"].apply(lambda p: Path(p).stem)
coincide = nombres.apply(lambda s: bool(patron.match(s)))
print(f"\nArchivos que siguen el patrón 'aug<id>_<sufijo>': {coincide.sum()} de {len(df)}")


## 8. Exploración gráfica adicional

### Imagen promedio por clase y distribución de intensidades

La imagen promedio permite ver qué forma "típica" tiene cada vocal; el histograma de intensidades ayuda a decidir si es necesario normalizar los píxeles.


In [ ]:
# Imagen promedio por clase
if not mismo_tamano:
    print("Las imágenes no tienen todas el mismo tamaño; se omite la imagen promedio.")
else:
    H, W = int(info["alto"].iloc[0]), int(info["ancho"].iloc[0])
    promedios = {}
    for c in CLASES:
        suma = np.zeros((H, W), dtype=np.float64)
        rutas = df.loc[df["clase"] == c, "ruta"]
        for r in rutas:
            with Image.open(r) as im:
                suma += np.array(im.convert("L"), dtype=np.float64)
        promedios[c] = suma / len(rutas)

    fig, axes = plt.subplots(1, len(CLASES), figsize=(3 * len(CLASES), 4))
    for ax, c in zip(axes, CLASES):
        ax.imshow(promedios[c], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"Promedio clase {c}")
        ax.axis("off")
    plt.suptitle("Imagen promedio por clase", y=1.02)
    plt.tight_layout()
    plt.show()

# Histograma de intensidades de píxel (muestra de 300 imágenes por clase)
fig, ax = plt.subplots(figsize=(8, 4))
for c in CLASES:
    sub = df[df["clase"] == c].sample(min(300, (df["clase"] == c).sum()), random_state=42)
    pix = np.concatenate([np.array(Image.open(r).convert("L")).ravel() for r in sub["ruta"]])
    ax.hist(pix, bins=64, range=(0, 255), alpha=0.4, density=True, label=c)
ax.set_title("Distribución de intensidades de píxel por clase")
ax.set_xlabel("Intensidad (0 = negro, 255 = blanco)")
ax.set_ylabel("Densidad")
ax.legend(title="Clase")
plt.tight_layout()
plt.show()


## 9. Resumen cuantitativo de la inspección


In [ ]:
resumen_final = pd.DataFrame({
    "Aspecto": [
        "Total de imágenes",
        "Número de clases",
        "Clases",
        "Imagen más/menos frecuente (razón)",
        "Formatos de archivo",
        "Tamaños distintos (ancho x alto)",
        "Modos de color",
        "Canales",
        "Imágenes corruptas",
        "Duplicados exactos",
        "Contenido repetido entre clases",
        "Imágenes con poco contraste",
    ],
    "Valor": [
        len(df),
        len(CLASES),
        ", ".join(CLASES),
        round(conteo.max() / conteo.min(), 3),
        ", ".join(df["formato"].unique()),
        "; ".join(f"{int(a)}x{int(b)}" for a, b in info[["ancho", "alto"]].drop_duplicates().values),
        ", ".join(info["modo"].unique()),
        ", ".join(str(int(x)) for x in info["canales"].dropna().unique()),
        len(corruptas),
        int(df["hash"].duplicated().sum()),
        int((df.groupby("hash")["clase"].nunique() > 1).sum()),
        len(poco_contraste),
    ],
})
resumen_final


## 10. Tratamientos propuestos para etapas posteriores

Con base en los hallazgos anteriores, se justifica qué tratamiento podría ser necesario más adelante (por ejemplo, redimensionado, conversión a escala de grises, normalización de intensidades, eliminación de duplicados o aumento de datos).
